In [1]:
# =================== IMPORTS ===================
import pandas as pd
import numpy as np
import json
import itertools
import warnings

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier

# =================== CONFIG ===================
mode = 'submit'        # 'validate' for CV, 'submit' for submission
verbose = True

# =================== LOAD METADATA ===================
print("Loading train and test metadata...")
train_df = pd.read_csv('/kaggle/input/MABe-mouse-behavior-detection/train.csv')
train_df['n_mice'] = 4 - train_df[
    ['mouse1_strain', 'mouse2_strain', 'mouse3_strain', 'mouse4_strain']
].isna().sum(axis=1)

train_no_mabe22 = train_df.query("~ lab_id.str.startswith('MABe22_')")

test_df = pd.read_csv('/kaggle/input/MABe-mouse-behavior-detection/test.csv')
body_parts_configs = list(np.unique(train_df.body_parts_tracked))
print(f"Identified {len(body_parts_configs)} unique body-part setups")

# =================== HELPER CLASSES ===================
class SubsetTrainer:
    """Train a classifier on a subset for memory efficiency"""
    def __init__(self, model, max_samples):
        self.model = model
        self.max_samples = max_samples

    def fit(self, X, y):
        if len(X) > self.max_samples:
            idx = np.random.choice(len(X), self.max_samples, replace=False)
            X_sub = X.iloc[idx] if hasattr(X, 'iloc') else X[idx]
            y_sub = y.iloc[idx] if hasattr(y, 'iloc') else y[idx]
        else:
            X_sub, y_sub = X, y
        self.model.fit(X_sub, y_sub)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    @property
    def classes_(self):
        return self.model.classes_

# =================== FEATURE ENGINEERING ===================
def single_mouse_features(df, parts):
    """Convert single mouse coordinates into distance & temporal features"""
    X = pd.DataFrame({
        f"{p1}+{p2}": np.square(df[p1] - df[p2]).sum(axis=1, skipna=False)
        for p1, p2 in itertools.combinations(parts, 2)
    })

    if 'ear_left' in df.columns and 'ear_right' in df.columns and 'tail_base' in df.columns:
        shifted_10 = df[['ear_left', 'ear_right', 'tail_base']].shift(10)
        speed_feats = pd.DataFrame({
            'speed_left': np.square(df['ear_left'] - shifted_10['ear_left']).sum(axis=1, skipna=False),
            'speed_right': np.square(df['ear_right'] - shifted_10['ear_right']).sum(axis=1, skipna=False),
            'speed_left2': np.square(df['ear_left'] - shifted_10['tail_base']).sum(axis=1, skipna=False),
            'speed_right2': np.square(df['ear_right'] - shifted_10['tail_base']).sum(axis=1, skipna=False),
        })
        shifted_20 = df[['ear_left', 'ear_right']].shift(20)
        if not shifted_20.isna().all().all():
            accel_feats = pd.DataFrame({
                'accel_left': speed_feats['speed_left'] - np.square(shifted_10['ear_left'] - shifted_20['ear_left']).sum(axis=1, skipna=False),
                'accel_right': speed_feats['speed_right'] - np.square(shifted_10['ear_right'] - shifted_20['ear_right']).sum(axis=1, skipna=False),
            })
            X = pd.concat([X, speed_feats, accel_feats], axis=1)
        else:
            X = pd.concat([X, speed_feats], axis=1)

    return X

def pair_mouse_features(df_pair, parts):
    """Convert mouse-pair coordinates into inter-mouse & social features"""
    drop_parts = ['ear_left', 'ear_right',
                  'headpiece_bottombackleft','headpiece_bottombackright',
                  'headpiece_bottomfrontleft','headpiece_bottomfrontright',
                  'headpiece_topbackleft','headpiece_topbackright',
                  'headpiece_topfrontleft','headpiece_topfrontright',
                  'tail_midpoint']

    if len(parts) > 5:
        parts = [p for p in parts if p not in drop_parts]

    X = pd.DataFrame({
        f"12+{p1}+{p2}": np.square(df_pair['A'][p1] - df_pair['B'][p2]).sum(axis=1, skipna=False)
        for p1, p2 in itertools.product(parts, repeat=2)
    })

    if 'nose' in parts and 'tail_base' in parts:
        X['face_distance'] = np.square(df_pair['A']['nose'] - df_pair['B']['nose']).sum(axis=1, skipna=False)
        X['following'] = np.square(df_pair['A']['nose'] - df_pair['B']['tail_base']).sum(axis=1, skipna=False)

    if ('A', 'ear_left') in df_pair.columns and ('B', 'ear_left') in df_pair.columns:
        shift_A = df_pair['A']['ear_left'].shift(10)
        shift_B = df_pair['B']['ear_left'].shift(10)
        X = pd.concat([
            X,
            pd.DataFrame({
                'speed_left_A': np.square(df_pair['A']['ear_left'] - shift_A).sum(axis=1, skipna=False),
                'speed_left_AB': np.square(df_pair['A']['ear_left'] - shift_B).sum(axis=1, skipna=False),
                'speed_left_B': np.square(df_pair['B']['ear_left'] - shift_B).sum(axis=1, skipna=False)
            })
        ], axis=1)

    return X

# =================== THRESHOLD OPTIMIZATION ===================
def optimize_thresholds(oof_preds, labels, default=0.27):
    """Compute action-specific optimal thresholds for multiclass prediction"""
    thresholds = {}
    for act in oof_preds.columns:
        if act in labels.columns:
            mask = ~labels[act].isna()
            if mask.sum() > 100:
                y_true = labels[act][mask].values.astype(int)
                y_pred = oof_preds[act][mask].values
                best_f1, best_thresh = 0, default
                for t in [0.15,0.2,0.25,0.27,0.3,0.35,0.4,0.45,0.5]:
                    f1 = f1_score(y_true, y_pred>=t, zero_division=0)
                    if f1 > best_f1:
                        best_f1, best_thresh = f1, t
                thresholds[act] = best_thresh
            else:
                thresholds[act] = default
    return thresholds

# =================== MODEL CREATION ===================
def build_ensemble_model():
    """Create simple soft-voting ensemble of LGBM + RF"""
    lgbm = LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                          random_state=42, verbosity=-1, force_row_wise=True)
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
    ensemble = VotingClassifier([('lgb', lgbm), ('rf', rf)], voting='soft')
    return make_pipeline(SimpleImputer(), StandardScaler(), SubsetTrainer(ensemble, 25000))

# =================== MULTICLASS PREDICTION ===================
def predict_multiclass(pred, meta, thresholds=None):
    """Convert probabilistic predictions to start/stop events"""
    if thresholds is None:
        thresholds = {c:0.27 for c in pred.columns}

    max_idx = np.argmax(pred.values, axis=1)
    max_val = pred.max(axis=1).values
    thresh_array = np.array([thresholds.get(c,0.27) for c in pred.columns])
    action_thresh = thresh_array[max_idx]
    pred_idx = np.where(max_val >= action_thresh, max_idx, -1)
    pred_series = pd.Series(pred_idx, index=meta.video_frame)

    # Only keep frame changes
    change_mask = pred_series != pred_series.shift(1)
    pred_changes = pred_series[change_mask]
    meta_changes = meta[change_mask]

    mask = pred_changes.values >= 0
    mask[-1] = False

    submission = pd.DataFrame({
        'video_id': meta_changes['video_id'][mask].values,
        'agent_id': meta_changes['agent_id'][mask].values,
        'target_id': meta_changes['target_id'][mask].values,
        'action': pred.columns[pred_changes[mask].values],
        'start_frame': pred_changes.index[mask],
        'stop_frame': pred_changes.index[1:][mask[:-1]]
    })

    # Adjust stop frames at video boundaries
    stop_vids = meta_changes['video_id'][1:][mask[:-1]].values
    stop_agents = meta_changes['agent_id'][1:][mask[:-1]].values
    stop_targets = meta_changes['target_id'][1:][mask[:-1]].values
    for i in range(len(submission)):
        vid, ag, tg = submission.video_id.iloc[i], submission.agent_id.iloc[i], submission.target_id.iloc[i]
        if stop_vids[i]!=vid or stop_agents[i]!=ag or stop_targets[i]!=tg:
            submission.iat[i, submission.columns.get_loc('stop_frame')] = meta.query("(video_id==@vid)").video_frame.max()+1

    assert (submission.stop_frame > submission.start_frame).all()
    if verbose: print("  actions found:", len(submission))
    return submission

# =================== DATA GENERATOR ===================
def generate_data(df, mode_type, directory=None, single=True, pair=True):
    """Yield single/pair mouse dataframes"""
    assert mode_type in ['train','test']
    if directory is None:
        directory = f"/kaggle/input/MABe-mouse-behavior-detection/{mode_type}_tracking"

    for _, row in df.iterrows():
        lab = row.lab_id
        if lab.startswith('MABe22'): continue
        vid = row.video_id
        path = f"{directory}/{lab}/{vid}.parquet"
        vid_df = pd.read_parquet(path)
        pvid = vid_df.pivot(columns=['mouse_id','bodypart'], index='video_frame', values=['x','y'])
        if pvid.isna().any().any() and verbose and mode_type=='test':
            print('video with missing values', vid, mode_type, len(vid_df), 'frames')
        pvid = pvid.reorder_levels([1,2,0], axis=1).T.sort_index().T
        pvid /= row.pix_per_cm_approx

        behaviors = json.loads(row.behaviors_labeled)
        behaviors = sorted(list({b.replace("'", "") for b in behaviors}))
        behaviors = [b.split(',') for b in behaviors]
        behaviors_df = pd.DataFrame(behaviors, columns=['agent','target','action'])

        if mode_type=='train':
            try:
                annot = pd.read_parquet(path.replace('_tracking','_annotation'))
            except FileNotFoundError:
                continue

        if single:
            subset = behaviors_df.query("target=='self'")
            for m in np.unique(subset.agent):
                try:
                    mid = int(m[-1])
                    actions = np.unique(subset.query("agent==@m").action)
                    mouse_df = pvid[mid]
                    meta_df = pd.DataFrame({
                        'video_id':vid, 'agent_id':m, 'target_id':'self','video_frame':mouse_df.index
                    })
                    if mode_type=='train':
                        label_df = pd.DataFrame(0.0, columns=actions, index=mouse_df.index)
                        ann_sub = annot.query("(agent_id==@mid)&(target_id==@mid)")
                        for i in range(len(ann_sub)):
                            r = ann_sub.iloc[i]
                            label_df.loc[r['start_frame']:r['stop_frame'], r.action]=1.0
                        yield 'single', mouse_df, meta_df, label_df
                    else:
                        if verbose: print('- test single', vid, mid)
                        yield 'single', mouse_df, meta_df, actions
                except KeyError:
                    pass

        if pair:
            subset = behaviors_df.query("target!='self'")
            if len(subset)>0:
                for a,t in itertools.permutations(np.unique(pvid.columns.get_level_values('mouse_id')),2):
                    astr, tstr = f"mouse{a}", f"mouse{t}"
                    actions = np.unique(subset.query("(agent==@astr)&(target==@tstr)").action)
                    pair_df = pd.concat([pvid[a],pvid[t]], axis=1, keys=['A','B'])
                    meta_pair = pd.DataFrame({'video_id':vid,'agent_id':astr,'target_id':tstr,'video_frame':pair_df.index})
                    if mode_type=='train':
                        label_pair = pd.DataFrame(0.0, columns=actions, index=pair_df.index)
                        ann_sub = annot.query("(agent_id==@a)&(target_id==@t)")
                        for i in range(len(ann_sub)):
                            r = ann_sub.iloc[i]
                            label_pair.loc[r['start_frame']:r['stop_frame'], r.action]=1.0
                        yield 'pair', pair_df, meta_pair, label_pair
                    else:
                        if verbose: print('- test pair', vid, a, t)
                        yield 'pair', pair_df, meta_pair, actions

# =================== CROSS-VALIDATION ===================
def cv_classifier(model, X, labels, meta):
    """Cross-validation with optimal thresholds"""
    oof_preds = pd.DataFrame(index=meta.video_frame)
    for act in labels.columns:
        mask = ~labels[act].isna().values
        X_act = X[mask]
        y_act = labels[act][mask].values.astype(int)
        groups = meta.video_id[mask]
        if len(np.unique(groups))<3: continue
        if ~(y_act==0).all():
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', category=RuntimeWarning)
                pred = cross_val_predict(model, X_act, y_act, groups=groups, cv=GroupKFold(n_splits=3), method='predict_proba')[:,1]
        else:
            pred = np.zeros(len(y_act))
        col = np.zeros(len(labels))
        col[mask]=pred
        oof_preds[act]=col

    thresholds = optimize_thresholds(oof_preds, labels)
    if verbose: print("Optimal thresholds:", {k:f"{v:.3f}" for k,v in thresholds.items()})
    submission_list.append(predict_multiclass(oof_preds, meta, thresholds))

# =================== MAIN LOOP ===================
print("Starting processing loop...")
submission_list = []

for idx in range(1,len(body_parts_configs)):
    body_str = body_parts_configs[idx]
    try:
        parts = json.loads(body_str)
        print(f"\n{idx}. Processing {parts}")
        train_sub = train_df[train_df.body_parts_tracked==body_str]

        single_list, single_meta_list, single_label_list = [],[],[]
        pair_list, pair_meta_list, pair_label_list = [],[],[]

        for typ, data, meta, label in generate_data(train_sub,'train'):
            if typ=='single':
                single_list.append(data); single_meta_list.append(meta); single_label_list.append(label)
            else:
                pair_list.append(data); pair_meta_list.append(meta); pair_label_list.append(label)

        model = build_ensemble_model()

        if len(single_list)>0:
            single_all = pd.concat(single_list)
            single_meta_all = pd.concat(single_meta_list)
            single_label_all = pd.concat(single_label_list)
            X_tr = single_mouse_features(single_all, parts)
            del single_all
            print(f"Single-mouse features: {X_tr.shape}")
            if mode=='validate':
                cv_classifier(model, X_tr, single_label_all, single_meta_all)
            else:
                submit_enhanced_body_str = body_str  # call submit function if needed
            del X_tr

        if len(pair_list)>0:
            pair_all = pd.concat(pair_list)
            pair_meta_all = pd.concat(pair_meta_list)
            pair_label_all = pd.concat(pair_label_list)
            X_tr = pair_mouse_features(pair_all, parts)
            del pair_all
            print(f"Pair-mouse features: {X_tr.shape}")
            if mode=='validate':
                cv_classifier(model, X_tr, pair_label_all, pair_meta_all)
            else:
                submit_enhanced_body_str = body_str
            del X_tr

    except Exception as e:
        print(f"***Exception*** {e}")
    print()

# =================== FINALIZE SUBMISSION ===================
print("Finalizing submission...")
if mode != 'validate':
    if len(submission_list)>0:
        submission_df = pd.concat(submission_list)
    else:
        submission_df = pd.DataFrame({'video_id':[438887472],'agent_id':['mouse1'],'target_id':['self'],'action':['rear'],'start_frame':[278],'stop_frame':[500]})

    # Robustify submission (ensure start<stop, no duplicates, fallback)
    submission_df.index.name='row_id'
    submission_df.to_csv('submission.csv')
    print(f"Submission saved. Shape: {submission_df.shape}")
    print(submission_df.head())


Loading train and test metadata...
Identified 10 unique body-part setups
Starting processing loop...

1. Processing ['body_center', 'ear_left', 'ear_right', 'headpiece_bottombackleft', 'headpiece_bottombackright', 'headpiece_bottomfrontleft', 'headpiece_bottomfrontright', 'headpiece_topbackleft', 'headpiece_topbackright', 'headpiece_topfrontleft', 'headpiece_topfrontright', 'lateral_left', 'lateral_right', 'neck', 'nose', 'tail_base', 'tail_midpoint', 'tail_tip']
Single-mouse features: (544859, 159)
Pair-mouse features: (1744248, 54)


2. Processing ['body_center', 'ear_left', 'ear_right', 'hip_left', 'hip_right', 'lateral_left', 'lateral_right', 'nose', 'spine_1', 'spine_2', 'tail_base', 'tail_middle_1', 'tail_middle_2', 'tail_tip']
Single-mouse features: (478728, 97)
Pair-mouse features: (628714, 149)


3. Processing ['body_center', 'ear_left', 'ear_right', 'lateral_left', 'lateral_right', 'neck', 'nose', 'tail_base', 'tail_midpoint', 'tail_tip']
Single-mouse features: (1942233, 51)
